In [1]:
!unzip archive.zip

Archive:  archive.zip
replace imagewoof200/test/australian_terrier/ILSVRC2012_val_00003690.JPEG? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [2]:
from pathlib import Path
data_path = Path("")
image_path = data_path / "imagewoof200"

In [3]:
import os
def check_data(dir_path):
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"# of directories in '{len(dirnames)}' and {len(filenames)} images in {dirpath}")

In [4]:
check_data(image_path)

# of directories in '2' and 0 images in imagewoof200
# of directories in '10' and 0 images in imagewoof200/test
# of directories in '0' and 48 images in imagewoof200/test/english_foxhound
# of directories in '0' and 48 images in imagewoof200/test/golden_retriever
# of directories in '0' and 48 images in imagewoof200/test/rhodesian_ridgeback
# of directories in '0' and 48 images in imagewoof200/test/shih_tzu
# of directories in '0' and 48 images in imagewoof200/test/dingo
# of directories in '0' and 48 images in imagewoof200/test/australian_terrier
# of directories in '0' and 48 images in imagewoof200/test/beagle
# of directories in '0' and 48 images in imagewoof200/test/border_terrier
# of directories in '0' and 48 images in imagewoof200/test/samoyed
# of directories in '0' and 48 images in imagewoof200/test/old_english_sheepdog
# of directories in '10' and 0 images in imagewoof200/train
# of directories in '0' and 200 images in imagewoof200/train/english_foxhound
# of directories in '

In [5]:
train_dir = image_path / "train"
test_dir  = image_path / "test"

In [6]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch

In [7]:
manual_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225])
])

In [8]:
NUM_WORKERS = os.cpu_count() or 0

def create_dataloader(train_dir: str,
                      test_dir: str,
                      transforms: transforms.Compose,
                      batch_size: int,
                      num_worksers: int = NUM_WORKERS):
    # Load training images from folder structure.
    train_data  = datasets.ImageFolder(root=train_dir,
                                       transform=transforms)

    # Load test images from folder structure.
    test_data = datasets.ImageFolder(root=test_dir,
                                     transform=transforms)

    # Get class names from training dataset folders.
    class_names = train_data.classes

    # Create DataLoader for training data (shuffled each epoch).
    train_dataloader = DataLoader(dataset=train_data,
                                  shuffle=True,
                                  batch_size=batch_size,
                                  num_workers=num_worksers)

    # Create DataLoader for test data (no shuffle for evaluation).
    test_dataloader =  DataLoader(dataset=test_data,
                                  batch_size=batch_size,
                                  shuffle=False,
                                  num_workers=num_worksers)

    # Return both dataloaders and class labels.
    return train_dataloader, test_dataloader, class_names

In [9]:
train_dataloader, test_dataloader, class_names = create_dataloader(train_dir=train_dir,
                                                                   test_dir=test_dir,
                                                                   transforms=manual_transforms,
                                                                   batch_size=32)

In [10]:
train_dataloader

In [11]:
class_names

['australian_terrier',
 'beagle',
 'border_terrier',
 'dingo',
 'english_foxhound',
 'golden_retriever',
 'old_english_sheepdog',
 'rhodesian_ridgeback',
 'samoyed',
 'shih_tzu']

In [12]:
import torchvision

In [24]:
weights = torchvision.models.EfficientNet_B3_Weights.DEFAULT

In [25]:
weights

EfficientNet_B3_Weights.IMAGENET1K_V1

In [26]:
auto_transforms = weights.transforms()

In [27]:
auto_transforms

ImageClassification(
    crop_size=[300]
    resize_size=[320]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [28]:
train_dataloader, test_dataloader, class_names = create_dataloader(train_dir=train_dir,
                                                                   test_dir=test_dir,
                                                                   transforms=auto_transforms,
                                                                   batch_size=32)

In [29]:
torch.cuda.is_available()

True

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [57]:
model = torchvision.models.efficientnet_b3(weights=weights).to(device)

In [58]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
            (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [59]:
!pip install torchinfo

In [60]:
from torchinfo import summary

In [61]:
summary(model,
        input_size=(32,3,224,224),
        col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 224, 224]         [32, 1000]                --                        True
├─Sequential: 1-1                                       [32, 3, 224, 224]         [32, 1536, 7, 7]          --                        True
│    └─Conv2dNormActivation: 2-1                        [32, 3, 224, 224]         [32, 40, 112, 112]        --                        True
│    │    └─Conv2d: 3-1                                 [32, 3, 224, 224]         [32, 40, 112, 112]        1,080                     True
│    │    └─BatchNorm2d: 3-2                            [32, 40, 112, 112]        [32, 40, 112, 112]        80                        True
│    │    └─SiLU: 3-3                                   [32, 40, 112, 112]        [32, 40, 112, 112]        --                        --
│    └─Sequential: 2-2  

In [62]:
for param in model.features.parameters():
  param.requires_grad = False

In [63]:
summary(model,
        input_size=(32,3,224,224),
        col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 224, 224]         [32, 1000]                --                        Partial
├─Sequential: 1-1                                       [32, 3, 224, 224]         [32, 1536, 7, 7]          --                        False
│    └─Conv2dNormActivation: 2-1                        [32, 3, 224, 224]         [32, 40, 112, 112]        --                        False
│    │    └─Conv2d: 3-1                                 [32, 3, 224, 224]         [32, 40, 112, 112]        (1,080)                   False
│    │    └─BatchNorm2d: 3-2                            [32, 40, 112, 112]        [32, 40, 112, 112]        (80)                      False
│    │    └─SiLU: 3-3                                   [32, 40, 112, 112]        [32, 40, 112, 112]        --                        --
│    └─Sequential

In [64]:
output_shape = len(class_names)

In [65]:
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1536,out_features=output_shape,)
    ).to(device)

In [66]:
summary(model,
        input_size=(32,3,224,224),
        col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 224, 224]         [32, 10]                  --                        Partial
├─Sequential: 1-1                                       [32, 3, 224, 224]         [32, 1536, 7, 7]          --                        False
│    └─Conv2dNormActivation: 2-1                        [32, 3, 224, 224]         [32, 40, 112, 112]        --                        False
│    │    └─Conv2d: 3-1                                 [32, 3, 224, 224]         [32, 40, 112, 112]        (1,080)                   False
│    │    └─BatchNorm2d: 3-2                            [32, 40, 112, 112]        [32, 40, 112, 112]        (80)                      False
│    │    └─SiLU: 3-3                                   [32, 40, 112, 112]        [32, 40, 112, 112]        --                        --
│    └─Sequential

In [67]:
from torch import nn

In [68]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=0.001)

In [69]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device
              ):
    model.train() # Modelimizi train moduna alıyoruz.

    train_loss = 0 # Train Loss değerlerini tutmak için bir değişken oluşturuyoruz.
    train_acc = 0 # Train Accuracy değerlerini tutmak için bir değişken oluşturuyoruz.

    for batch, (X,y) in enumerate(dataloader): # Batch size gerekli değil burada.
        X,y = X.to(device), y.to(device)
        y_pred = model(X) # Modelimize bir tahminde bulunduruyoruz.

        loss = loss_fn(y_pred,y) # Loss değerlerimizi loss_fn ile hesaplıyoruz.
        train_loss += loss.item() # Çıkan loss değerlerini train_loss değişlenine toplayarak atıyoruz.

        # Modelimizi backpropagation yapıyoruz.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Softmax kullanarak modelimize label tahmininde bulunduruyoruz.
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

        train_acc += (y_pred_class == y).sum().item() / len(y_pred) # Accuracy değerlerimizi bir değişkende tutuyoruz.

    train_loss /= len(dataloader) # Train Loss değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz.
    train_acc /= len(dataloader) # Train Acc değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz
    return train_loss, train_acc # Geriye train_loss ve train_acc döndürüyoruz.

def test_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
              device: torch.device
              ):
    model.eval() # Modelimizi test moduna alıyoruz.

    test_loss = 0 # test loss'ları tutmak için bir değişken oluşturuyoruz.
    test_acc = 0 # test accuracy'ları tutmak için bir değişken oluşturuyoruz.

    with torch.inference_mode(): # inference mode'a aldık.
        for batch, (X,y) in enumerate(dataloader): # batch gerekli değil fakat yine de aldık.
            X,y = X.to(device), y.to(device)
            test_pred = model(X) # modelimize tahmin ettiriyoruz.

            loss = loss_fn(test_pred,y) # loss'umuzu loss_fn ile hesaplıyoruz.
            test_loss += loss.item() # loss değerlerimizi test_loss değişkeninde topluyoruz.

            # Softmax activation function ile label tahmin ettiriyoruz.
            test_pred_label = torch.softmax(test_pred,dim=1).argmax(dim=1)

            acc = (test_pred_label == y).sum().item() / len(test_pred) # Calculate accuracy
            test_acc += acc # Accuracy değerlerimizi toplayıp test_acc değişkenine atıyoruz.

    test_loss /= len(dataloader) # Test loss değerlerimizi dataloader boyuna bölüyoruz.
    test_acc /= len(dataloader) # Test acc değerlerimizi dataloader boyuna bölüyoruz.

    return test_loss, test_acc # Geriye test_loss ve test_acc döndürüyoruz.

def train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               device:torch.device,
               loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
               epochs:int = 10,
              ):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer,
                                           device = device
                                          )
        test_loss, test_acc = test_step(model = model,
                                           dataloader = test_dataloader,
                                           loss_fn = loss_fn,
                                           device = device
                                          )
        print(f"""
        Epoch:{epoch}
        Train Loss : {train_loss:.2f} -  Train Accuracy : {train_acc*100:.2f}
        Test Loss  : { test_loss:.2f} -  Test Accuracy  : {test_acc*100:.2f}
        """)
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)
    return results


In [70]:
results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn = loss_fn,
                device = device,
                epochs=10)

 
        Epoch:0
        Train Loss : 1.31 -  Train Accuracy : 76.29
        Test Loss  : 0.65 -  Test Accuracy  : 93.33
        
 
        Epoch:1
        Train Loss : 0.57 -  Train Accuracy : 90.38
        Test Loss  : 0.40 -  Test Accuracy  : 92.92
        
 
        Epoch:2
        Train Loss : 0.40 -  Train Accuracy : 91.72
        Test Loss  : 0.33 -  Test Accuracy  : 93.33
        
 
        Epoch:3
        Train Loss : 0.34 -  Train Accuracy : 92.31
        Test Loss  : 0.31 -  Test Accuracy  : 92.71
        
 
        Epoch:4
        Train Loss : 0.28 -  Train Accuracy : 93.70
        Test Loss  : 0.28 -  Test Accuracy  : 92.29
        
 
        Epoch:5
        Train Loss : 0.25 -  Train Accuracy : 94.35
        Test Loss  : 0.26 -  Test Accuracy  : 93.33
        
 
        Epoch:6
        Train Loss : 0.24 -  Train Accuracy : 94.35
        Test Loss  : 0.26 -  Test Accuracy  : 92.92
        
 
        Epoch:7
        Train Loss : 0.22 -  Train Accuracy : 95.14
        Test 